# Kepler exoplanet classification — Notebook 03

**Deep learning models: MLP, ResNet, and FT-Transformer**

Author: Atilla Ahmed

---

## Abstract

This notebook implements and evaluates three modern deep learning architectures for tabular data on the Kepler classification task: a Multi-Layer Perceptron (MLP), a residual-connection tabular network (ResNet-Tabular), and a Feature Tokenizer + Transformer (FT-Transformer).
Models are trained primarily on the leak-free feature configuration defined in Notebook 01 and benchmarked against the XGBoost baseline established in Notebook 02. A soft-voting ensemble combines the three architectures for the final result.

## Table of contents

1. [Data loading and training setup](#1-data-loading-and-training-setup)
2. [Multi-Layer Perceptron](#2-multi-layer-perceptron)
3. [Residual Tabular Network](#3-residual-tabular-network)
4. [Feature Tokenizer Transformer](#4-feature-tokenizer-transformer)
5. [Model ensemble](#5-model-ensemble)
6. [Summary](#6-summary)

## 1. Data loading and training setup

We load the processed splits from Notebook 01, prepare PyTorch tensors on the GPU, and define a single training function that will be used consistently for all three deep learning models.

### 1.1. Imports and configuration

In [1]:
import json
import warnings
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
BATCH_SIZE = 256
MAX_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

PROCESSED_PATH = Path("../data/processed")
MODELS_PATH = Path("../models/dl")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

int_to_class = {0: "CONFIRMED", 1: "CANDIDATE", 2: "FALSE POSITIVE"}
class_order = ["CONFIRMED", "CANDIDATE", "FALSE POSITIVE"]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Setup complete")

Setup complete


### 1.2. GPU configuration

We detect the available compute device and prepare a global `device` variable used consistently across all model definitions and training loops.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

Device: cuda
GPU:    NVIDIA GeForce RTX 4050 Laptop GPU


### 1.3. Load processed data

We read the three parquet splits, load the feature-set definitions, and apply the same `StandardScaler` fit on training data used in Notebook 02. Since PyTorch models cannot participate in an `sklearn.pipeline.Pipeline` cleanly, we scale the features explicitly before creating tensors.

In [3]:
train_df = pd.read_parquet(PROCESSED_PATH / "train.parquet")
val_df   = pd.read_parquet(PROCESSED_PATH / "val.parquet")
test_df  = pd.read_parquet(PROCESSED_PATH / "test.parquet")

with open(PROCESSED_PATH / "feature_sets.json", "r") as f:
    feature_sets = json.load(f)

def split_features_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    return df.drop(columns=["target"]), df["target"]

X_train, y_train = split_features_target(train_df)
X_val,   y_val   = split_features_target(val_df)
X_test,  y_test  = split_features_target(test_df)

feature_sets_effective = {
    "setup_leaky":     [c for c in feature_sets["setup_a_leaky"] if c in X_train.columns],
    "setup_leak_free": [c for c in feature_sets["setup_c_leak_free"] if c in X_train.columns],
}

print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
for name, features in feature_sets_effective.items():
    print(f"  {name}: {len(features)} features")

Train:      (6694, 101)
Validation: (1435, 101)
Test:       (1435, 101)
  setup_leaky: 55 features
  setup_leak_free: 51 features
